In [1]:
import pandas as pd
import pyterrier as pt
from ir_measures import *
import math


if not pt.started():
    pt.init()

/tmp/ipykernel_6723/1850020821.py:7: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_6723/1850020821.py:8: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [2]:
ohsumed_docs = pd.read_pickle("/nfs/primary/sas_reranker/ohsumed_docs_w_t5base_sensitivity.pkl")
queries = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/queries.pkl")
qrels = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/qrels.pkl")

In [3]:
def _sens_docs(qrels, run):
    if "sensitivity" in run.columns:
        run = run.drop(columns = ["sensitivity"])
    merged = pd.merge(run, ohsumed_docs, left_on = "doc_id", right_on = "docno")
    return merged.sensitivity.sum()
    
import ir_measures
sens_docs = ir_measures.define_byquery(
    _sens_docs, 
    name="sens_docs")

def calculate_csdcg_10(qrels, run):
    if "sensitivity" in run.columns:
        run = run.drop(columns = ["sensitivity"])
    merged = pd.merge(run, qrels, on = "doc_id", how = "left").fillna(0).head(10)
    merged = pd.merge(merged, ohsumed_docs, left_on = "doc_id", right_on = "docno")
    # print(merged[["relevance", "sensitivity"]])
    csdcg = 0
    for rank, doc in enumerate(merged.iterrows()):
        relevance = int(doc[1].relevance)
        sensitivity = int(doc[1]["sensitivity"])
        up = (2 ** relevance) - 1
        low = math.log(rank + 2)
        penalty = 12 if sensitivity == 1 else 0
        csdcg += ((up / low) - penalty)
    return csdcg

def _ncsdcg_cut_10(qrels, run):    
    # Calculate query csdcg
    query_csdcg = calculate_csdcg_10(qrels, run)
    
    # Calculate best csdcg
    merged_qrels = pd.merge(qrels, ohsumed_docs, left_on = "doc_id", right_on = "docno").drop(columns = ["title", "abstract", "text", "t5_sensitivity", "docno"])
    best_qrels =  merged_qrels.sort_values(by=['sensitivity', 'relevance'], ascending=[True, False])
    best_qrels = best_qrels.drop(columns = ["relevance", "sensitivity"])
    best_csdcg = calculate_csdcg_10(qrels, best_qrels)
    
    # Calculate worst csdcg
    worst_qrels =  merged_qrels.sort_values(by=['sensitivity', 'relevance'], ascending=[False, True])
    worst_qrels = worst_qrels.drop(columns = ["relevance", "sensitivity"])
    worst_csdcg = calculate_csdcg_10(qrels, worst_qrels)
    
    if query_csdcg <= worst_csdcg:
        query_csdcg = 0
        
    normalised_csdcg = (query_csdcg - worst_csdcg) / (best_csdcg - worst_csdcg)

    assert normalised_csdcg >= 0
    assert normalised_csdcg <= 1
    
    return normalised_csdcg
    
import ir_measures
nCSDCG_cut_10 = ir_measures.define_byquery(
    _ncsdcg_cut_10, 
    name="nCSDCG_cut_10")

def _ns_recall(qrels, run):
    combined = pd.merge(run, qrels, on = ["doc_id"])
    combined = pd.merge(combined, ohsumed_docs, left_on = "doc_id", right_on = "docno")
    combined.sensitivity = combined.sensitivity.astype(int)
    combined.relevance = combined.relevance.astype(int)
    num_rel_ns = combined[(combined['sensitivity'] < 1) & (combined['relevance'] > 1)].shape[0]
    qrels_w_sensitivity = pd.merge(qrels, ohsumed_docs, left_on = "doc_id", right_on = "docno")
    total = qrels_w_sensitivity[(qrels_w_sensitivity['sensitivity'] < 1) & (qrels_w_sensitivity['relevance'] > 1)].shape[0]
    assert num_rel_ns <= total
    if total == 0:
        return 0
    return num_rel_ns / total
    
import ir_measures
NSR = ir_measures.define_byquery(
    _ns_recall, 
    name="NSR")

In [4]:
def create_pairwise_significance_df(results_paths, queries, qrels, eval_metrics, names, 
                                  correction="b", round_digits=4):
    """
    Create a dataframe showing pairwise statistical significance between all systems.
    
    Parameters:
    -----------
    results_paths : list
        List of paths to result files
    queries : pyterrier queries
        Query dataframe
    qrels : pyterrier qrels
        Relevance judgments
    eval_metrics : list
        List of evaluation metrics
    names : list
        List of system names
    correction : str
        Multiple testing correction method (default: "b" for Bonferroni)
    round_digits : int
        Number of decimal places for rounding (default: 4)
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: name and each metric as separate columns
        Metric values have significance indicators (a,b,c,d,...) appended
        where each letter indicates significance against system at that index
    """
    
    # Read all results
    all_results = [pt.io.read_results(path) for path in results_paths]
    
    # First get the actual metric values by running one experiment
    base_exp = pt.Experiment(
        all_results,
        queries,
        qrels,
        eval_metrics=eval_metrics,
        names=names,
        round=round_digits,
        baseline=0,
        correction=correction
    )
    
    # Extract actual metric column names from the experiment result
    # Filter out non-metric columns (name, statistical test columns, etc.)
    metric_names = []
    for col in base_exp.columns:
        if (col != 'name' and 
            not col.endswith(' +') and 
            not col.endswith(' -') and 
            not col.endswith(' p-value') and 
            not col.endswith(' reject') and 
            not col.endswith(' p-value corrected')):
            metric_names.append(col)
    
    # Initialize results storage
    pairwise_results = []
    
    # Run pairwise comparisons
    for baseline_idx in range(len(names)):
        # Create experiment with current system as baseline
        exp = pt.Experiment(
            all_results,
            queries,
            qrels,
            eval_metrics=eval_metrics,
            names=names,
            round=round_digits,
            baseline=baseline_idx,
            correction=correction
        )
        
        # Extract significance results for each system and metric
        for system_idx, system_name in enumerate(names):
            for metric_name in metric_names:
                # Get the reject column for this metric
                reject_col = f"{metric_name} reject"
                
                if reject_col in exp.columns and system_idx < len(exp):
                    is_significant = exp.iloc[system_idx][reject_col]
                    
                    pairwise_results.append({
                        'system': system_name,
                        'metric': metric_name,
                        'baseline': names[baseline_idx],
                        'baseline_idx': baseline_idx,
                        'is_significant': is_significant if pd.notna(is_significant) else False
                    })
    
    # Convert to DataFrame for easier manipulation
    pairwise_df = pd.DataFrame(pairwise_results)
    
    # Create the final summary DataFrame with metric columns
    summary_data = []
    
    for system_idx, system_name in enumerate(names):
        row_data = {'name': system_name}
        
        for metric_name in metric_names:
            # Get the base metric value
            metric_value = base_exp.iloc[system_idx][metric_name]
            
            # Get significance results for this system-metric combination
            system_metric_data = pairwise_df[
                (pairwise_df['system'] == system_name) & 
                (pairwise_df['metric'] == metric_name)
            ]
            
            # Create significance indicators (a, b, c, d, ...)
            significance_indicators = []
            for i in range(len(names)):
                baseline_name = names[i]
                sig_row = system_metric_data[system_metric_data['baseline'] == baseline_name]
                
                if len(sig_row) > 0 and sig_row.iloc[0]['is_significant']:
                    significance_indicators.append(chr(ord('a') + i))
            
            # Append significance indicators to metric value
            significance_str = ''.join(significance_indicators)
            metric_with_sig = f"{metric_value} {significance_str}" if significance_str else str(metric_value)
            
            row_data[metric_name] = metric_with_sig
        
        summary_data.append(row_data)
    
    return pd.DataFrame(summary_data)


# Example usage with your data:
def run_analysis(queries, qrels, eval_metrics, results_paths, names):
    """
    Run the pairwise analysis with your specific data.
    Replace this with your actual queries, qrels, and eval_metrics.
    """

    # Run the pairwise comparison
    significance_df = create_pairwise_significance_df(
        results_paths=results_paths,
        queries=queries,
        qrels=qrels, 
        eval_metrics=eval_metrics,
        names=names,
        correction="b",
        round_digits=4
    )
    
    return significance_df

In [5]:
exp = pt.Experiment(
    [
        pt.io.read_results("./runs/bm25 % 100"),
        pt.io.read_results("./runs/splade_relevance % 100"),
        pt.io.read_results("./runs/splade_sans % 100"),
        pt.io.read_results("./runs/splade_distil % 100")
    ],
    queries,
    qrels,
    eval_metrics=[nDCG@10, sens_docs@10, nCSDCG_cut_10],
    names = [
        "BM25",
        "SPLADE Relevance",
        "SPLADE SANS",
        "SPLADE Distil"
    ],
    round = 4
)

exp

,name,nDCG@10,sens_docs@10,nCSDCG_cut_10
0,BM25,0.3917,1.3396,0.7077
1,SPLADE Relevance,0.4375,1.2170,0.7276
2,SPLADE SANS,0.4557,0.8302,0.7848
3,SPLADE Distil,0.2841,0.2736,0.7983


In [6]:
exp = pt.Experiment(
    [
        pt.io.read_results("./runs/intermediates/bm25 % 100 >> filter % 100"),
        pt.io.read_results("./runs/intermediates/splade_relevance % 100 >> filter % 100"),
        pt.io.read_results("./runs/intermediates/splade_sans % 100 >> filter % 100"),
        pt.io.read_results("./runs/intermediates/splade_distil % 100 >> filter % 100"),
    ],
    queries,
    qrels,
    eval_metrics=[nDCG@10, sens_docs@10, nCSDCG_cut_10],
    names = [
        "BM25 >> Filter",
        "SPLADE Relevance >> Filter",
        "SPLADE SANS >> Filter",
        "SPLADE Distil >> Filter"
    ],
    round = 4
)

exp

,name,nDCG@10,sens_docs@10,nCSDCG_cut_10
0,BM25 >> Filter,0.3839,1.1698,0.7250
1,SPLADE Relevance >> Filter,0.4289,1.0283,0.7462
2,SPLADE SANS >> Filter,0.4461,0.7925,0.7869
3,SPLADE Distil >> Filter,0.2833,0.2170,0.8033


In [7]:
exp = pt.Experiment(
    [
        pt.io.read_results("./runs/intermediates/bm25 % 100 >> filter % 100 >> get_text >> monot5_relevance"),
        pt.io.read_results("./runs/intermediates/splade_relevance % 100 >> filter % 100 >> get_text >> monot5_relevance"),
        pt.io.read_results("./runs/intermediates/splade_sans % 100 >> filter % 100 >> get_text >> monot5_relevance"),
        pt.io.read_results("./runs/intermediates/splade_distil % 100 >> filter % 100 >> get_text >> monot5_relevance"),
    ],
    queries,
    qrels,
    eval_metrics=[nDCG@10, sens_docs@10, nCSDCG_cut_10],
    names = [
        "BM25 >> Filter >> monoT5 (Relevance)",
        "SPLADE Relevance >> Filter >> monoT5 (Relevance)",
        "SPLADE SANS >> Filter >> monoT5 (Relevance)",
        "SPLADE Distil >> Filter >> monoT5 (Relevance)"
    ],
    round = 4
)

exp

,name,nDCG@10,sens_docs@10,nCSDCG_cut_10
0,BM25 >> Filter >> monoT5 (Relevance),0.4706,1.2453,0.7327
1,SPLADE Relevance >> Filter >> monoT5 (Relevance),0.4802,1.1981,0.7618
2,SPLADE SANS >> Filter >> monoT5 (Relevance),0.4842,1.0755,0.7683
3,SPLADE Distil >> Filter >> monoT5 (Relevance),0.4505,0.5283,0.8148


In [8]:
exp = pt.Experiment(
    [
        pt.io.read_results("./runs/intermediates/bm25 % 100 >> filter % 100 >> get_text >> monot5_sans"),
        pt.io.read_results("./runs/intermediates/splade_relevance % 100 >> filter % 100 >> get_text >> monot5_sans"),
        pt.io.read_results("./runs/intermediates/splade_sans % 100 >> filter % 100 >> get_text >> monot5_sans"),
        pt.io.read_results("./runs/intermediates/splade_distil % 100 >> filter % 100 >> get_text >> monot5_sans"),
    ],
    queries,
    qrels,
    eval_metrics=[nDCG@10, sens_docs@10, nCSDCG_cut_10],
    names = [
        "BM25 >> Filter >> monoT5 (SANS)",
        "SPLADE Relevance >> Filter >> monoT5 (SANS)",
        "SPLADE SANS >> Filter >> monoT5 (SANS)",
        "SPLADE Distil >> Filter >> monoT5 (SANS)"
    ],
    round = 4,
)

exp

,name,nDCG@10,sens_docs@10,nCSDCG_cut_10
0,BM25 >> Filter >> monoT5 (SANS),0.3724,0.6604,0.7797
1,SPLADE Relevance >> Filter >> monoT5 (SANS),0.4237,0.6887,0.7975
2,SPLADE SANS >> Filter >> monoT5 (SANS),0.4173,0.7075,0.7897
3,SPLADE Distil >> Filter >> monoT5 (SANS),0.3957,0.3585,0.8193


<h1>Perform Significance Testing </h1>

In [9]:
results_paths = ["./runs/bm25 % 100", "./runs/splade_relevance % 100",  "./runs/splade_sans % 100", "./runs/splade_distil % 100"]
names = ["BM25 % 100", "SPLADE Relevance % 100", "SPLADE SANS % 100", "SPLADE Distil % 100"]
metrics = [nDCG@10, sens_docs@10, nCSDCG_cut_10, nDCG@100, sens_docs@100, NSR@100]

run_analysis(queries, qrels, metrics, results_paths, names)

,name,nCSDCG_cut_10,sens_docs@100,NSR@100,sens_docs@10,nDCG@10,nDCG@100
0,BM25 % 100,0.7077 cd,13.4906 cd,0.5985 d,1.3396 cd,0.3917 cd,0.4582 cd
1,SPLADE Relevance % 100,0.7276 cd,12.4906 cd,0.5566 d,1.217 cd,0.4375 d,0.4661 cd
2,SPLADE SANS % 100,0.7848 ab,9.3491 abd,0.5926 d,0.8302 abd,0.4557 ad,0.5063 abd
3,SPLADE Distil % 100,0.7983 ab,2.6792 abc,0.3941 abc,0.2736 abc,0.2841 abc,0.3039 abc


In [10]:
results_paths = ["./runs/intermediates/bm25 % 100 >> filter % 100", "./runs/intermediates/splade_relevance % 100 >> filter % 100", "./runs/intermediates/splade_sans % 100 >> filter % 100", "./runs/intermediates/splade_distil % 100 >> filter % 100"]    
names = ["BM25  %100 >> Filter", "SPLADE Relevance % 100 >> Filter", "SPLADE SANS % 100 >> Filter", "SPLADE Distil % 100 >> Filter"]
metrics = [nDCG@10, sens_docs@10, nCSDCG_cut_10, nDCG@100, sens_docs@100, NSR@100]

run_analysis(queries, qrels, metrics, results_paths, names)

,name,nCSDCG_cut_10,sens_docs@100,NSR@100,sens_docs@10,nDCG@10,nDCG@100
0,BM25 %100 >> Filter,0.725 cd,9.3396 cd,0.5795 d,1.1698 cd,0.3839 cd,0.441 cd
1,SPLADE Relevance % 100 >> Filter,0.7462 cd,8.7264 cd,0.5367 cd,1.0283 d,0.4289 d,0.4494 cd
2,SPLADE SANS % 100 >> Filter,0.7869 ab,7.1887 abd,0.5826 bd,0.7925 ad,0.4461 ad,0.4923 abd
3,SPLADE Distil % 100 >> Filter,0.8033 ab,1.9811 abc,0.3829 abc,0.217 abc,0.2833 abc,0.3007 abc


In [11]:
results_paths = ["./runs/intermediates/bm25 % 100 >> filter % 100 >> get_text >> monot5_relevance", "./runs/intermediates/splade_relevance % 100 >> filter % 100 >> get_text >> monot5_relevance", "./runs/intermediates/splade_sans % 100 >> filter % 100 >> get_text >> monot5_relevance", "./runs/intermediates/splade_distil % 100 >> filter % 100 >> get_text >> monot5_relevance",]
names = [ "BM25 % 100>> Filter >> monoT5 (Relevance)", "SPLADE Relevance % 100 >> Filter >> monoT5 (Relevance)", "SPLADE SANS % 100 >> Filter >> monoT5 (Relevance)", "SPLADE Distil % 100 >> Filter >> monoT5 (Relevance)"]

metrics = [nDCG@10, sens_docs@10, nCSDCG_cut_10, nDCG@100, sens_docs@100, NSR@100]

run_analysis(queries, qrels, metrics, results_paths, names)

,name,nCSDCG_cut_10,sens_docs@100,NSR@100,sens_docs@10,nDCG@10,nDCG@100
0,BM25 % 100>> Filter >> monoT5 (Relevance),0.7327 cd,9.3396 cd,0.5795 d,1.2453 cd,0.4706,0.473 cd
1,SPLADE Relevance % 100 >> Filter >> monoT5 (Re...,0.7618 d,8.7264 cd,0.5367 cd,1.1981 d,0.4802,0.4728 cd
2,SPLADE SANS % 100 >> Filter >> monoT5 (Relevance),0.7683 ad,7.1887 abd,0.5826 bd,1.0755 ad,0.4842 d,0.5079 abd
3,SPLADE Distil % 100 >> Filter >> monoT5 (Relev...,0.8148 abc,1.9811 abc,0.3829 abc,0.5283 abc,0.4505 c,0.3687 abc


In [12]:
results_paths =  ["./runs/intermediates/bm25 % 100 >> filter % 100 >> get_text >> monot5_sans", "./runs/intermediates/splade_relevance % 100 >> filter % 100 >> get_text >> monot5_sans", "./runs/intermediates/splade_sans % 100 >> filter % 100 >> get_text >> monot5_sans", "./runs/intermediates/splade_distil % 100 >> filter % 100 >> get_text >> monot5_sans",]
names = ["BM25 >> Filter >> monoT5 (SANS)","SPLADE Relevance >> Filter >> monoT5 (SANS)","SPLADE SANS >> Filter >> monoT5 (SANS)","SPLADE Distil >> Filter >> monoT5 (SANS)"]
metrics = [nDCG@10, sens_docs@10, nCSDCG_cut_10, nDCG@100, sens_docs@100, NSR@100]

run_analysis(queries, qrels, metrics, results_paths, names)

,name,nCSDCG_cut_10,sens_docs@100,NSR@100,sens_docs@10,nDCG@10,nDCG@100
0,BM25 >> Filter >> monoT5 (SANS),0.7797 d,9.3396 cd,0.5795 d,0.6604 d,0.3724 bc,0.4415 cd
1,SPLADE Relevance >> Filter >> monoT5 (SANS),0.7975,8.7264 cd,0.5367 cd,0.6887 d,0.4237 a,0.4598 cd
2,SPLADE SANS >> Filter >> monoT5 (SANS),0.7897,7.1887 abd,0.5826 bd,0.7075 d,0.4173 a,0.4881 abd
3,SPLADE Distil >> Filter >> monoT5 (SANS),0.8193 a,1.9811 abc,0.3829 abc,0.3585 abc,0.3957,0.3509 abc


In [14]:
relevance_paths = ["./runs/intermediates/bm25 % 100 >> filter % 100 >> get_text >> monot5_relevance", "./runs/intermediates/splade_relevance % 100 >> filter % 100 >> get_text >> monot5_relevance", "./runs/intermediates/splade_sans % 100 >> filter % 100 >> get_text >> monot5_relevance", "./runs/intermediates/splade_distil % 100 >> filter % 100 >> get_text >> monot5_relevance",]
relevance_names = [ "BM25 >> Filter >> monoT5 (Relevance)", "SPLADE Relevance >> Filter >> monoT5 (Relevance)", "SPLADE SANS >> Filter >> monoT5 (Relevance)", "SPLADE Distil >> Filter >> monoT5 (Relevance)"]
sans_paths =  ["./runs/intermediates/bm25 % 100 >> filter % 100 >> get_text >> monot5_sans", "./runs/intermediates/splade_relevance % 100 >> filter % 100 >> get_text >> monot5_sans", "./runs/intermediates/splade_sans % 100 >> filter % 100 >> get_text >> monot5_sans", "./runs/intermediates/splade_distil % 100 >> filter % 100 >> get_text >> monot5_sans",]
sans_names = ["BM25 >> Filter >> monoT5 (SANS)","SPLADE Relevance >> Filter >> monoT5 (SANS)","SPLADE SANS >> Filter >> monoT5 (SANS)","SPLADE Distil >> Filter >> monoT5 (SANS)"]

results_paths = relevance_paths + sans_paths
names = relevance_names + sans_names

metrics = [nDCG@10, sens_docs@10, nCSDCG_cut_10, nDCG@100, sens_docs@100, NSR@100]

run_analysis(queries, qrels, metrics, results_paths, names)

,name,nCSDCG_cut_10,sens_docs@100,NSR@100,sens_docs@10,nDCG@10,nDCG@100
0,BM25 >> Filter >> monoT5 (Relevance),0.7327 defgh,9.3396 cdgh,0.5795 dh,1.2453 defgh,0.4706 eh,0.473 deh
1,SPLADE Relevance >> Filter >> monoT5 (Relevance),0.7618 dh,8.7264 cdgh,0.5367 cdgh,1.1981 defgh,0.4802 efgh,0.4728 cdh
2,SPLADE SANS >> Filter >> monoT5 (Relevance),0.7683 dh,7.1887 abdefh,0.5826 bdfh,1.0755 defgh,0.4842 efgh,0.5079 bdefh
3,SPLADE Distil >> Filter >> monoT5 (Relevance),0.8148 abc,1.9811 abcefg,0.3829 abcefg,0.5283 abch,0.4505 eh,0.3687 abcefg
4,BM25 >> Filter >> monoT5 (SANS),0.7797 ah,9.3396 cdgh,0.5795 dh,0.6604 abc,0.3724 abcdfg,0.4415 acdgh
5,SPLADE Relevance >> Filter >> monoT5 (SANS),0.7975 a,8.7264 cdgh,0.5367 cdgh,0.6887 abc,0.4237 bce,0.4598 cdgh
6,SPLADE SANS >> Filter >> monoT5 (SANS),0.7897 a,7.1887 abdefh,0.5826 bdfh,0.7075 abc,0.4173 bce,0.4881 defh
7,SPLADE Distil >> Filter >> monoT5 (SANS),0.8193 abce,1.9811 abcefg,0.3829 abcefg,0.3585 abcd,0.3957 abcd,0.3509 abcefg
